In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd


In [ ]:
# Load the dataset
df_bcn = pd.read_csv('data/barcelona_2015_2026.csv', sep=',', engine='python')
df_temp = pd.read_csv('data/barcelona_temp_2020_2025_test.csv', sep=',', engine='python')
df_bcn_ampliado = pd.read_csv('data/BCN_sentinel_completo_2020_2025.csv', sep=',', engine='python')


In [ ]:
df_temp.describe().T

In [ ]:
df_bcn_ampliado.describe().T

In [ ]:
df_bcn.head()

PIPELINE FECHA

In [ ]:
df_temp['fecha'] = pd.to_datetime(df_temp['fecha'])
df_bcn['fecha'] = pd.to_datetime(df_bcn['fecha'])

In [ ]:
fechas_comunes = set(df_temp['fecha']).intersection(
    set(df_bcn['fecha'])
)

df_temp = df_temp[df_temp['fecha'].isin(fechas_comunes)]
df_bcn = df_bcn[df_bcn['fecha'].isin(fechas_comunes)]

In [ ]:
gdf_temp = gpd.GeoDataFrame(
    df_temp,
    geometry=gpd.points_from_xy(
        df_temp.longitude,
        df_temp.latitude
    ),
    crs='EPSG:4326'
)

gdf_bcn = gpd.GeoDataFrame(
    df_bcn,
    geometry=gpd.points_from_xy(
        df_bcn.longitude,
        df_bcn.latitude
    ),
    crs='EPSG:4326'
)

gdf_bcn_ampliado = gpd.GeoDataFrame(
    df_bcn_ampliado,
    geometry=gpd.points_from_xy(
        df_bcn_ampliado.longitude,
        df_bcn_ampliado.latitude
    ),
    crs='EPSG:4326'
)

In [ ]:
gdf_temp = gdf_temp.to_crs(32631)
gdf_bcn = gdf_bcn.to_crs(32631)
gdf_bcn_ampliado = gdf_bcn_ampliado.to_crs(32631)

In [ ]:
gdf_bcn_ampliado.info()

In [ ]:
gdf_temp.info()

In [ ]:
merged_list = []

for fecha in fechas_comunes:

    temp_day = gdf_temp[gdf_temp['fecha'] == fecha]

    bcn_day = gdf_bcn[gdf_bcn['fecha'] == fecha]

    merged_day = gpd.sjoin_nearest(
        bcn_day,
        temp_day[
            [
                'lst_day_c',
                'lst_night_c',
                'geometry'
            ]
        ],
        how='left',
        distance_col='dist_m'
    )

    merged_list.append(merged_day)

dataset_final = pd.concat(
    merged_list,
    ignore_index=True
)

In [ ]:
from pyproj import Transformer

GRID_SIZE = 1000
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)

# Añadir grid_id a gdf_temp
x, y = transformer.transform(
    gdf_temp['longitude'].values,
    gdf_temp['latitude'].values
)

gdf_temp['x']       = x
gdf_temp['y']       = y
gdf_temp['grid_x']  = (x // GRID_SIZE).astype(int)
gdf_temp['grid_y']  = (y // GRID_SIZE).astype(int)
gdf_temp['grid_id'] = (
    gdf_temp['grid_x'].astype(str)
    + '_'
    + gdf_temp['grid_y'].astype(str)
)

print(gdf_temp['grid_id'].head(3).tolist())
print(gdf_bcn_ampliado['grid_id'].head(3).tolist())

In [ ]:
grids_modis    = set(gdf_temp['grid_id'].unique())
grids_sentinel = set(gdf_bcn_ampliado['grid_id'].unique())

comunes = grids_modis.intersection(grids_sentinel)

print(f'Grids MODIS:    {len(grids_modis)}')
print(f'Grids Sentinel: {len(grids_sentinel)}')
print(f'Grids comunes:  {len(comunes)}')

In [ ]:
# ============================================================
# AGREGAR SENTINEL A NIVEL GRID
# ============================================================
sentinel_cols = [c for c in [
    'ndvi','ndbi','elevation',
    'evi','evi2','savi','msavi','gndvi','ndre','cire','gli','psri',
    'ibi','ui','ebbi','bsi','mndwi','ndwi','ndmi','ndsi','nbr'
] if c in gdf_bcn_ampliado.columns]

sentinel_agg = (
    gdf_bcn_ampliado
    .groupby(['fecha', 'grid_id'])[sentinel_cols]
    .mean()
    .reset_index()
)

# ============================================================
# AGREGAR MODIS A NIVEL GRID
# ============================================================
modis_agg = (
    gdf_temp
    .groupby(['fecha', 'grid_id'])[['lst_day_c', 'lst_night_c']]
    .mean()
    .reset_index()
)

sentinel_agg['fecha'] = pd.to_datetime(sentinel_agg['fecha'])
modis_agg['fecha']    = pd.to_datetime(modis_agg['fecha'])

dataset_final = pd.merge(
    sentinel_agg,
    modis_agg,
    on=['fecha', 'grid_id'],
    how='inner'
)

print(f'Dataset final: {dataset_final.shape}')
print(f'Fechas únicas: {dataset_final["fecha"].nunique()}')
print(f'Grids únicos:  {dataset_final["grid_id"].nunique()}')
# ============================================================
# JOIN
# ============================================================
dataset_final = pd.merge(
    sentinel_agg,
    modis_agg,
    on=['fecha', 'grid_id'],
    how='inner'
)

# ============================================================
# VARIABLES TEMPORALES + ANOMALÍAS
# ============================================================
dataset_final['month']     = dataset_final['fecha'].dt.month
dataset_final['year']      = dataset_final['fecha'].dt.year
dataset_final['dayofyear'] = dataset_final['fecha'].dt.dayofyear

dataset_final['lst_day_c_anomaly'] = (
    dataset_final['lst_day_c']
    - dataset_final.groupby('month')['lst_day_c'].transform('mean')
)

dataset_final['lst_night_c_anomaly'] = (
    dataset_final['lst_night_c']
    - dataset_final.groupby('month')['lst_night_c'].transform('mean')
)

print(f'Sentinel agg:  {sentinel_agg.shape}')
print(f'MODIS agg:     {modis_agg.shape}')
print(f'Dataset final: {dataset_final.shape}')
print(f'\nFechas únicas: {dataset_final["fecha"].nunique()}')
print(f'Grids únicos:  {dataset_final["grid_id"].nunique()}')
dataset_final.head()

In [ ]:
# ============================================================
# JOIN POR fecha_modis + grid_id
# ============================================================
modis_agg['fecha'] = pd.to_datetime(modis_agg['fecha'])

# Renombrar fecha en modis_agg para evitar colisión
modis_agg_temp = modis_agg.rename(columns={'fecha': 'fecha_modis'})

dataset_final = pd.merge(
    sentinel_agg_matched,
    modis_agg_temp,
    on=['fecha_modis', 'grid_id'],
    how='inner'
)

# Renombrar fecha Sentinel como fecha principal
dataset_final = dataset_final.rename(columns={'fecha': 'fecha_sentinel'})

print(f'Dataset final: {dataset_final.shape}')
print(f'Fechas Sentinel únicas: {dataset_final["fecha_sentinel"].nunique()}')
print(f'Fechas MODIS únicas:    {dataset_final["fecha_modis"].nunique()}')
print(f'Grids únicos:           {dataset_final["grid_id"].nunique()}')

In [ ]:
import numpy as np
from pyproj import Transformer

TOLERANCIA_DIAS = 15

# ============================================================
# MATCH FECHAS CON TOLERANCIA
# ============================================================
fechas_sentinel = pd.to_datetime(sentinel_agg['fecha'].unique())
fechas_modis    = pd.to_datetime(modis_agg['fecha'].unique())

# Para cada fecha Sentinel, encontrar la fecha MODIS más cercana
fecha_map = {}
for fs in fechas_sentinel:
    diffs = np.abs((fechas_modis - fs).days)
    idx_min = diffs.argmin()
    if diffs[idx_min] <= TOLERANCIA_DIAS:
        fecha_map[fs] = fechas_modis[idx_min]

print(f'Fechas Sentinel con match: {len(fecha_map)}/24')
print('\nMapping fechas:')
for k, v in fecha_map.items():
    print(f'  Sentinel {k.date()} → MODIS {v.date()}')

# ============================================================
# APLICAR MAPPING A SENTINEL AGG
# ============================================================
sentinel_agg['fecha_modis'] = sentinel_agg['fecha'].map(fecha_map)

# Descartar fechas sin match
sentinel_agg_matched = sentinel_agg.dropna(subset=['fecha_modis']).copy()
sentinel_agg_matched['fecha_modis'] = pd.to_datetime(sentinel_agg_matched['fecha_modis'])

print(f'\nSentinel agg matched: {sentinel_agg_matched.shape}')

# ============================================================
# JOIN POR fecha_modis + grid_id
# ============================================================
modis_agg['fecha'] = pd.to_datetime(modis_agg['fecha'])

dataset_final = pd.merge(
    sentinel_agg_matched,
    modis_agg,
    left_on=['fecha_modis', 'grid_id'],
    right_on=['fecha', 'grid_id'],
    how='inner',
    suffixes=('_sentinel', '_modis')
)

# Limpiar columnas de fecha duplicadas
dataset_final = dataset_final.drop(columns=['fecha_modis', 'fecha_modis'])
dataset_final = dataset_final.rename(columns={'fecha_sentinel': 'fecha'})

# ============================================================
# VARIABLES TEMPORALES + ANOMALÍAS
# ============================================================
dataset_final['month']     = dataset_final['fecha'].dt.month
dataset_final['year']      = dataset_final['fecha'].dt.year
dataset_final['dayofyear'] = dataset_final['fecha'].dt.dayofyear

dataset_final['lst_day_c_anomaly'] = (
    dataset_final['lst_day_c']
    - dataset_final.groupby('month')['lst_day_c'].transform('mean')
)

dataset_final['lst_night_c_anomaly'] = (
    dataset_final['lst_night_c']
    - dataset_final.groupby('month')['lst_night_c'].transform('mean')
)

print(f'\nDataset final: {dataset_final.shape}')
print(f'Fechas únicas: {dataset_final["fecha"].nunique()}')
print(f'Grids únicos:  {dataset_final["grid_id"].nunique()}')
dataset_final.head()

In [ ]:
dataset_final['month']     = dataset_final['fecha_sentinel'].dt.month
dataset_final['year']      = dataset_final['fecha_sentinel'].dt.year
dataset_final['dayofyear'] = dataset_final['fecha_sentinel'].dt.dayofyear

dataset_final['lst_day_c_anomaly'] = (
    dataset_final['lst_day_c']
    - dataset_final.groupby('month')['lst_day_c'].transform('mean')
)

dataset_final['lst_night_c_anomaly'] = (
    dataset_final['lst_night_c']
    - dataset_final.groupby('month')['lst_night_c'].transform('mean')
)

# Guardar
dataset_final.to_csv('data/barcelona_dataset_final.csv', index=False)

print(f'Shape: {dataset_final.shape}')
print(f'Columnas: {dataset_final.columns.tolist()}')
print(dataset_final.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
dataset_final = dataset_final.rename(columns={'fecha_sentinel': 'fecha'})
dataset_final.to_csv('data/barcelona_dataset_final.csv', index=False)

In [ ]:
dataset_final.shape

LIMPIEZA

In [ ]:
dataset_final = dataset_final.drop(
    columns=[
        'geometry',
        'index_right'
    ],
    errors='ignore'
)

In [ ]:
dataset_final.info()

In [ ]:
#MAtriz de correlación entre ndvi, ndbi, lst_day_c y lst_night_c
corr_matrix = dataset_final[
    [
        'ndvi',
        'ndbi',
        'lst_day_c',
        'lst_night_c'
    ]
].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')


In [ ]:
#R2 de pearson entre ndvi y lst_day_c. y entre ndbi y lst_day_c
pearson_corr_ndvi_lst_day = dataset_final['ndvi'].corr(dataset_final['lst_day_c'])
print(f'Correlación de Pearson entre NDVI y LST Day: {pearson_corr_ndvi_lst_day}')

pearson_corr_ndbi_lst_day = dataset_final['ndbi'].corr(dataset_final['lst_day_c'])
print(f'Correlación de Pearson entre NDBI y LST Day: {pearson_corr_ndbi_lst_day}')

SPLIT + TRAIN

In [ ]:
df_temp.groupby('fecha').size().describe()

In [ ]:
df_temp.groupby('fecha').size().sort_values().head(20)

Filtar por imágenes con suficientes valores válidos

GRID 1 km

In [ ]:
import geopandas as gpd
import pandas as pd

gdf_bcn = gpd.GeoDataFrame(
    df_bcn,
    geometry=gpd.points_from_xy(
        df_bcn.longitude,
        df_bcn.latitude
    ),
    crs='EPSG:4326'
)

gdf_temp = gpd.GeoDataFrame(
    df_temp,
    geometry=gpd.points_from_xy(
        df_temp.longitude,
        df_temp.latitude
    ),
    crs='EPSG:4326'
)

In [ ]:
gdf_bcn = gdf_bcn.to_crs(32631)
gdf_temp = gdf_temp.to_crs(32631)

In [ ]:
gdf_bcn['x'] = gdf_bcn.geometry.x
gdf_bcn['y'] = gdf_bcn.geometry.y

gdf_temp['x'] = gdf_temp.geometry.x
gdf_temp['y'] = gdf_temp.geometry.y

In [ ]:
GRID_SIZE = 1000

gdf_bcn['grid_x'] = (
    gdf_bcn['x'] // GRID_SIZE
).astype(int)

gdf_bcn['grid_y'] = (
    gdf_bcn['y'] // GRID_SIZE
).astype(int)

gdf_bcn['grid_id'] = (
    gdf_bcn['grid_x'].astype(str)
    + '_'
    + gdf_bcn['grid_y'].astype(str)
)

In [ ]:
gdf_temp['grid_x'] = (
    gdf_temp['x'] // GRID_SIZE
).astype(int)

gdf_temp['grid_y'] = (
    gdf_temp['y'] // GRID_SIZE
).astype(int)

gdf_temp['grid_id'] = (
    gdf_temp['grid_x'].astype(str)
    + '_'
    + gdf_temp['grid_y'].astype(str)
)

In [ ]:
sentinel_agg = (
    gdf_bcn
    .groupby(['fecha', 'grid_id'])
    .agg({
        'ndvi': 'mean',
        'ndbi': 'mean',
        'elevation': 'mean',
        'latitude': 'mean',
        'longitude': 'mean'
    })
    .reset_index()
)

In [ ]:
modis_agg = (
    gdf_temp
    .groupby(['fecha', 'grid_id'])
    .agg({
        'lst_day_c': 'mean',
        'lst_night_c': 'mean'
    })
    .reset_index()
)

In [ ]:
#MERGE
dataset_final = pd.merge(
    sentinel_agg,
    modis_agg,
    on=['fecha', 'grid_id'],
    how='inner'
)

In [ ]:
dataset_final.shape

In [ ]:
dataset_final.groupby('fecha').size().describe()

In [ ]:
dataset_final['fecha'] = pd.to_datetime(
    dataset_final['fecha']
)

dataset_final['month'] = (
    dataset_final['fecha'].dt.month
)

dataset_final['dayofyear'] = (
    dataset_final['fecha'].dt.dayofyear
)

dataset_final['year'] = (
    dataset_final['fecha'].dt.year
)

VALIDEZ

In [ ]:
dataset_final.isna().mean()

In [ ]:
dataset_final['lst_day_c'].describe()

In [ ]:
#Crear columna de anomalia términa para lst_day_c restando la media mensual de lst_day_c a cada valor de lst_day_c
dataset_final['lst_day_c_anomaly'] = (dataset_final['lst_day_c'] - dataset_final['lst_day_c'].mean())

In [ ]:
#media de temperatura 
dataset_final['lst_day_c'].mean()

In [ ]:
#Crear columna de anomalia térmica para lst_night_c restando la media mensual de lst_night_c a cada valor de lst_night_c
dataset_final['lst_night_c_anomaly'] = (dataset_final['lst_night_c'] - dataset_final['lst_night_c'].mean())

In [ ]:
#histograma de lst_night_c_anomaly
sns.histplot(dataset_final['lst_night_c_anomaly'], bins=30, kde=True)

In [ ]:
#histograma de lst_day_c_anomaly
sns.histplot(dataset_final['lst_day_c_anomaly'], bins=30, kde=True)

In [ ]:
#Clasificar lst_day_c_anomaly en 3 categorias: baja, media y alta
dataset_final['lst_day_c_anomaly_cat'] = pd.cut(dataset_final['lst_day_c_anomaly'], bins=3, labels=['baja', 'media', 'alta'])

In [ ]:
dataset_final['lst_day_c_anomaly_cat'].value_counts()

In [ ]:
dataset_final.describe()

In [ ]:
dataset_final.groupby('fecha').size().describe()

In [ ]:
print(dataset_final['fecha'].dt.strftime('%Y-%m-%d').unique().tolist())

RANDOM FOREST DE PRUEBA

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare Features and Target
features = ['ndvi', 'ndbi', 'elevation', 'latitude', 'longitude']
target = 'lst_day_c_anomaly_cat'

# Ensure the categorical column is ready (handled in your previous steps)
# Using the same temporal split defined in your notebook
train = dataset_final[dataset_final['year'] <= 2023]
test = dataset_final[dataset_final['year'] >= 2024]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# 2. Initialize and Train the Classifier
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced' # Handles potential imbalance in anomaly categories
)

rf_classifier.fit(X_train, y_train)

# 3. Make Predictions
y_pred = rf_classifier.predict(X_test)

# 4. Evaluation
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 5. Visualizing Feature Importance
importances = pd.Series(rf_classifier.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=importances.index, palette='viridis')
plt.title('Feature Importance for Heat Anomaly Classification')
plt.show()

# 6. Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=['baja', 'media', 'alta'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['baja', 'media', 'alta'], yticklabels=['baja', 'media', 'alta'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix: Anomaly Class')
plt.show()

In [ ]:
#Distribución visual de fechas en dataset_final
dataset_final['fecha'].value_counts().sort_index().plot(kind='line')
plt.xlabel('Fecha')


In [ ]:
#Timeseries que muestre la evolución de lst_day_c a lo largo del tiempo
dataset_final.groupby('fecha')['lst_day_c'].mean().plot(kind='line')

In [ ]:
train = dataset_final[
    dataset_final['year'] <= 2023
]

test = dataset_final[
    dataset_final['year'] >= 2024
]

In [ ]:
features = [
    'ndvi',
    'ndbi',
    'elevation',
    'latitude',
    'longitude',
    # 'dayofyear'
]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf.fit(
    train[features],
    train['lst_day_c']
)

In [ ]:
preds = rf.predict(test[features])

In [ ]:
print(test['lst_day_c'])

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import numpy as np

mse = mean_squared_error(
    test['lst_day_c'],
    preds
)

rmse = np.sqrt(mse)

print('R2:', r2_score(test['lst_day_c'], preds))
print('MAE:', mean_absolute_error(test['lst_day_c'], preds))
print('RMSE:', rmse)

In [ ]:
import pandas as pd

importance = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

In [ ]:
residuals = test['lst_day_c'] - preds
sns.scatterplot(x=preds, y=residuals)
plt.axhline(0, color='red', linestyle='--')

JOIN_GEO

In [ ]:
gdf_bcn_ndvi = gpd.GeoDataFrame(
    df_bcn,
    geometry=gpd.points_from_xy(
        df_bcn.longitude,
        df_bcn.latitude
    ),
    crs="EPSG:4326"
)

gdf_bcn_ndvi = gdf_bcn_ndvi.to_crs(secciones.crs)

In [ ]:
gdf_joined = gpd.sjoin(
    gdf_bcn_ndvi,
    secciones_bcn,
    how="inner",
    predicate="within"
)

In [ ]:
sentinel_agg = (
    gdf_joined
    .groupby(['CUSEC'])
    .agg({
        'ndvi': 'mean',
        'ndbi': 'mean',
        'elevation': 'mean',
        'latitude': 'mean',
        'longitude': 'mean'
    })
    .reset_index()
)

In [ ]:
print(sentinel_agg.head())
print(sentinel_agg.shape)
sentinel_agg.info()